# Supervised Machine Learning Model - Loan Default Prediction

**Author:** Juana Zhang  
**Dataset:** `data/sample.csv` (not included in this public portfolio repository)  
**Target Variable:** `loan_status`

## Technical Guidelines

- Use `loan_status` as the target variable.
- Treat the remaining columns as independent variables, excluding the `id` column.
- Begin with a baseline Logistic Regression model.
- Progress to more advanced techniques, including tree-based models.

## Modeling Approach

This notebook uses a reproducible modeling pipeline while keeping the steps visible for review:

1. Setup and validate the dataset.
2. Define target, predictors, and train/test split.
3. Build preprocessing and evaluation logic.
4. Define baseline and advanced model candidates.
5. Select the best model using cross-validation on `train_full`.
6. Tune the threshold with out-of-fold predictions and evaluate on test.
7. Interpret the selected model and save outputs.
8. Summarize the modeling conclusion.

## 1. Setup and Data Validation

In [ ]:
from pathlib import Path
import os
import json
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data" / "sample.csv"
OUTPUT_DIR = Path.cwd().parent / "outputs"
FIGURE_DIR = Path.cwd().parent / "figures"
OUTPUT_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)

**Load and validate the source data.**

In [ ]:
df = pd.read_csv(DATA_PATH)

required_columns = {"id", "loan_status"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

print("Dataset shape:", df.shape)
print("Missing values:", df.isna().sum().sum())
print("Duplicated IDs:", df["id"].duplicated().sum())
df.head()

## 2. Target, Predictors, and Train/Test Split

Following the assignment requirement, `loan_status` is the target variable and `id` is excluded from the independent variables. All remaining non-ID variables are retained as candidate predictors.

In [ ]:
X = df.drop(columns=["id", "loan_status"])
y = df["loan_status"]

categorical_features = X.select_dtypes(include=["object", "string"]).columns.tolist()
numeric_features = X.select_dtypes(exclude=["object", "string"]).columns.tolist()

print("Target variable: loan_status")
print("Excluded column: id")
print("Predictor count:", X.shape[1])
print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

**Train/test split.**

Because the target is imbalanced, a stratified split is used so the default rate is represented consistently across training and test data.

The workflow is:

- `train_full`: cross-validation model selection, out-of-fold threshold tuning, and final model fitting.
- `test`: final holdout performance reporting only.

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

split_summary = pd.DataFrame(
    {
        "dataset": ["train_full", "test"],
        "rows": [len(X_train_full), len(X_test)],
        "default_rate": [y_train_full.mean(), y_test.mean()],
    }
)
split_summary

## 3. Preprocessing and Evaluation Design

Numerical and categorical variables are handled separately:

- Numerical features are scaled for Logistic Regression.
- Numerical features are passed through directly for tree-based models.
- All categorical features are retained and encoded with one-hot encoding.

In [ ]:
def build_preprocessor(scale_numeric: bool) -> ColumnTransformer:
    numeric_transformer = StandardScaler() if scale_numeric else "passthrough"
    categorical_transformer = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ],
        remainder="drop",
        sparse_threshold=0,
    )

**Evaluation functions.**

In [ ]:
def evaluate_model(name: str, estimator: Pipeline, X_eval: pd.DataFrame, y_eval: pd.Series) -> dict:
    probabilities = estimator.predict_proba(X_eval)[:, 1]
    predictions = (probabilities >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_eval, predictions).ravel()
    return {
        "model": name,
        "threshold": 0.5,
        "roc_auc": roc_auc_score(y_eval, probabilities),
        "pr_auc": average_precision_score(y_eval, probabilities),
        "accuracy": accuracy_score(y_eval, predictions),
        "precision": precision_score(y_eval, predictions, zero_division=0),
        "recall": recall_score(y_eval, predictions, zero_division=0),
        "f1": f1_score(y_eval, predictions, zero_division=0),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def find_best_f1_threshold(y_true: pd.Series, probabilities: np.ndarray):
    precision, recall, thresholds = precision_recall_curve(y_true, probabilities)
    f1_scores = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
    best_idx = int(np.nanargmax(f1_scores[:-1]))
    threshold = float(thresholds[best_idx])
    predictions = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions).ravel()
    return threshold, {
        "threshold": threshold,
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "f1": f1_score(y_true, predictions, zero_division=0),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

## 4. Candidate Models

The modeling sequence follows the assignment:

1. Baseline Logistic Regression.
2. Tree-based Random Forest.
3. Gradient Boosting model.

In [ ]:
models = {
    "Logistic Regression": Pipeline(
        steps=[
            ("preprocess", build_preprocessor(scale_numeric=True)),
            (
                "model",
                LogisticRegression(
                    max_iter=2000,
                    class_weight="balanced",
                    solver="lbfgs",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
    "Random Forest": Pipeline(
        steps=[
            ("preprocess", build_preprocessor(scale_numeric=False)),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=350,
                    min_samples_leaf=10,
                    class_weight="balanced_subsample",
                    random_state=RANDOM_STATE,
                    n_jobs=1,
                ),
            ),
        ]
    ),
    "Gradient Boosting": Pipeline(
        steps=[
            ("preprocess", build_preprocessor(scale_numeric=False)),
            (
                "model",
                HistGradientBoostingClassifier(
                    max_iter=300,
                    learning_rate=0.06,
                    l2_regularization=0.03,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
}

## 5. Cross-Validation Model Selection

Model selection is based on cross-validation performance on `train_full`. Models are compared by cross-validated PR-AUC, with supporting metrics and visualization used to show relative performance. PR-AUC is emphasized because the positive default class is underrepresented and ranking quality matters more than accuracy alone.

The test set is not used for model selection or threshold tuning.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
trained_models = {}

for name, pipeline in models.items():
    print(f"Cross-validating {name}...")
    cv_result = cross_validate(
        pipeline,
        X_train_full,
        y_train_full,
        cv=cv,
        scoring=["roc_auc", "average_precision", "f1", "recall", "precision"],
        n_jobs=1,
    )
    cv_rows.append(
        {
            "model": name,
            "cv_roc_auc_mean": cv_result["test_roc_auc"].mean(),
            "cv_roc_auc_std": cv_result["test_roc_auc"].std(),
            "cv_pr_auc_mean": cv_result["test_average_precision"].mean(),
            "cv_pr_auc_std": cv_result["test_average_precision"].std(),
            "cv_f1_mean": cv_result["test_f1"].mean(),
            "cv_recall_mean": cv_result["test_recall"].mean(),
            "cv_precision_mean": cv_result["test_precision"].mean(),
        }
    )

    pipeline.fit(X_train_full, y_train_full)
    trained_models[name] = pipeline

cv_metrics = pd.DataFrame(cv_rows).sort_values("cv_pr_auc_mean", ascending=False)
display(cv_metrics.round(4))

cv_plot_data = cv_metrics.melt(
    id_vars="model",
    value_vars=["cv_roc_auc_mean", "cv_pr_auc_mean", "cv_f1_mean"],
    var_name="metric",
    value_name="score",
)
cv_plot_data["metric"] = cv_plot_data["metric"].map(
    {
        "cv_roc_auc_mean": "CV ROC-AUC",
        "cv_pr_auc_mean": "CV PR-AUC",
        "cv_f1_mean": "CV F1",
    }
)

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(
    data=cv_plot_data,
    x="model",
    y="score",
    hue="metric",
    ax=ax,
    palette=["#4C78A8", "#F58518", "#54A24B"],
)
ax.set_title("Cross-Validated Model Performance")
ax.set_xlabel("")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", rotation=0)
ax.legend(title="")
plt.tight_layout()
plt.show()

## 6. Threshold Tuning and Final Test Evaluation

After the best model is selected, the remaining steps are based only on that selected model.

First, out-of-fold predicted probabilities from `train_full` are used to optimize the classification threshold. The objective is to maximize F1, creating a more balanced tradeoff between precision and recall.

Finally, model performance is reported on the independent holdout test set. The test set is not involved in model selection or threshold optimization, which helps avoid data leakage.

In [ ]:
best_model_name = cv_metrics.iloc[0]["model"]
best_model = trained_models[best_model_name]
print("Best model selected by CV PR-AUC:", best_model_name)

oof_probabilities = cross_val_predict(
    models[best_model_name],
    X_train_full,
    y_train_full,
    cv=cv,
    method="predict_proba",
    n_jobs=1,
)[:, 1]
tuned_threshold, oof_threshold_metrics = find_best_f1_threshold(y_train_full, oof_probabilities)

test_probabilities = best_model.predict_proba(X_test)[:, 1]
test_predictions_tuned = (test_probabilities >= tuned_threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, test_predictions_tuned).ravel()
tuned_test_metrics = {
    "model": best_model_name,
    "threshold": tuned_threshold,
    "roc_auc": roc_auc_score(y_test, test_probabilities),
    "pr_auc": average_precision_score(y_test, test_probabilities),
    "accuracy": accuracy_score(y_test, test_predictions_tuned),
    "precision": precision_score(y_test, test_predictions_tuned, zero_division=0),
    "recall": recall_score(y_test, test_predictions_tuned, zero_division=0),
    "f1": f1_score(y_test, test_predictions_tuned, zero_division=0),
    "tn": int(tn),
    "fp": int(fp),
    "fn": int(fn),
    "tp": int(tp),
}

selected_model_test_metrics = pd.DataFrame([tuned_test_metrics])
metrics = selected_model_test_metrics.copy()

display(pd.DataFrame([oof_threshold_metrics]).round(4).assign(dataset="OOF train_full"))
display(selected_model_test_metrics.round(4).assign(dataset="test"))

## 7. Feature Importance and Saved Outputs

In [ ]:
perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=8,
    random_state=RANDOM_STATE,
    n_jobs=1,
)

feature_importance = (
    pd.DataFrame(
        {
            "feature": X.columns,
            "importance": perm.importances_mean,
            "importance_std": perm.importances_std,
        }
    )
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

display(feature_importance.round(4))

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(
    data=feature_importance.head(12),
    y="feature",
    x="importance",
    ax=ax,
    color="#4C78A8",
)
ax.set_title("Top Predictive Drivers")
ax.set_xlabel("Permutation Importance - ROC AUC Decrease")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

**Save outputs.**

The selected model, model comparison tables, feature importance table, and summary JSON are saved for reproducibility and reporting.

In [ ]:
metrics.to_csv(OUTPUT_DIR / "model_metrics.csv", index=False)
cv_metrics.to_csv(OUTPUT_DIR / "cross_validation_metrics.csv", index=False)
feature_importance.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)
joblib.dump(best_model, OUTPUT_DIR / "best_model.joblib")

categorical_rates = {}
for col in categorical_features:
    categorical_rates[col] = (
        df.groupby(col)["loan_status"]
        .agg(count="count", default_rate="mean")
        .sort_values("default_rate", ascending=False)
        .reset_index()
        .to_dict(orient="records")
    )

summary = {
    "data": {
        "rows": int(df.shape[0]),
        "columns": int(df.shape[1]),
        "target_positive_count": int(y.sum()),
        "target_negative_count": int((1 - y).sum()),
        "target_positive_rate": float(y.mean()),
        "missing_values_total": int(df.isna().sum().sum()),
        "duplicate_ids": int(df["id"].duplicated().sum()),
        "duplicate_rows_excluding_id": int(df.drop(columns=["id"]).duplicated().sum()),
    },
    "feature_setup": {
        "target": "loan_status",
        "excluded_columns": ["id"],
        "predictor_count": int(X.shape[1]),
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
        "categorical_encoding": "OneHotEncoder(handle_unknown='ignore')",
    },
    "numerical_summary": df.drop(columns=["id", "loan_status"]).describe().T.round(4).to_dict(orient="index"),
    "categorical_default_rates": categorical_rates,
    "target_correlations": df.drop(columns=["id"]).corr(numeric_only=True)["loan_status"]
        .drop("loan_status")
        .sort_values(key=lambda s: s.abs(), ascending=False)
        .round(6)
        .to_dict(),
    "model_metrics": metrics.to_dict(orient="records"),
    "cross_validation_metrics": cv_metrics.to_dict(orient="records"),
    "best_model": best_model_name,
    "oof_threshold_metrics": oof_threshold_metrics,
    "tuned_test_metrics": tuned_test_metrics,
    "feature_importance": feature_importance.to_dict(orient="records"),
}

with open(OUTPUT_DIR / "modeling_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved outputs to:", OUTPUT_DIR.resolve())

## 8. Modeling Conclusion

The baseline Logistic Regression model provides a transparent starting point, while the tree-based models capture additional nonlinear structure. Based on cross-validated PR-AUC, Gradient Boosting is selected as the best-performing model. The decision threshold is tuned using out-of-fold predictions from `train_full`, and final performance is reported on the untouched holdout test set. The strongest model drivers are consistent with the EDA narrative: affordability, housing status, pricing, income, and loan purpose are central risk signals.